In [0]:
# databricks-feature-engineering==0.7.0
# databricks-sdk==0.38.0

In [0]:
from pyspark.sql import SparkSession, DataFrame
import pyspark.sql.types as T
import yaml
import os
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from datetime import datetime

from typing import Dict
import yaml
import logging

In [0]:
class LoggingUtils:
    def __init__(self, spark: SparkSession):
        """
        input: spark (SparkSession)
        description: Initialize LoggingUtils with a SparkSession.
        output: None
        """
        self.spark = spark

    ####
    def get_variable(self) -> Dict:
        """
        input: None
        description: Returns a dictionary of configuration and metadata paths.
        output: Dict containing paths.
        """
    def get_variable(self, product_name) -> Dict:
        """
        input: None
        description: Returns a dictionary of configuration and metadata paths.
        output: Dict containing paths.
        """
        return {
            "config_path": f"/Workspace/AI/{product_name}/{product_name}_model/files/workflow/configs/config.yml",
            "config_data_path": f"/Workspace/AI/{product_name}/{product_name}_model/files/src/config/config_data.yml",
            "config_setup_path": f"/Workspace/AI/{product_name}/{product_name}_model/files/src/config/config_setup.yml",
            "config_sys_path": f"/Workspace/AI/{product_name}/{product_name}_model/files",
            "metadata_path": "p_ind_core_ai.metadata.md_ai_object",
            "databricks_asset_path": f"/Workspace/AI/{product_name}/{product_name}_model/files/databricks.yml"
            }
    
    ####
    def load_config(self, config_path=None):
        """
        input: config_path (str, optional)
        description: Loads YAML configuration from the given path or environment variable.
        output: Parsed YAML configuration as dict.
        """
        if config_path is None:
            config_path = os.getenv('ARCHIVER_CONFIG_PATH', 'config.yaml')
        with open(config_path, 'r') as f:
            return yaml.safe_load(f)
    
    ####
    def get_log_processor_reference(self, catalog: str, schema: str, table: str):
        """
        input: catalog (str), schema (str), table (str)
        description: Returns catalog, schema, and table reference.
        output: Tuple of (catalog, schema, table)
        """
        return catalog, schema, table

    ####
    def populate_monitor_log_date(self, catalog: str, database: str, table: str, metrica: str, cod_runID: str, is_started: bool):
        """
        input: catalog (str), database (str), table (str), metrica (str), cod_runID (str), is_started (bool)
        description: Updates process start or end date in the log table.
        output: None
        """
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        full_table_path = f"{catalog}.{database}.{table}"
        
        target_column = "dat_processStartDate" if is_started else "dat_processEndDate"
        
        sql_query = f"""
            UPDATE {full_table_path}
            SET {target_column} = CAST('{now}' AS TIMESTAMP)
            WHERE cod_flowName = '{metrica}' 
              AND cod_runID = '{cod_runID}'
              AND ({target_column} IS NULL OR CAST({target_column} AS STRING) = 'null')
        """
        self.spark.sql(sql_query)

    ####
    def populate_log_error(self, catalog: str, database: str, table: str, metrica: str, snapshot_date: str, result_state: str, cod_runID: str, is_error: bool):
        """
        input: catalog (str), database (str), table (str), metrica (str), snapshot_date (str), result_state (str), cod_runID (str), is_error (bool)
        description: Updates error code in the log table if an error occurred.
        output: None
        """
        if not is_error: return
        
        full_table_path = f"{catalog}.{database}.{table}"
        sql_query = f"""
            UPDATE {full_table_path}
            SET cod_processErrorCode = '{result_state}'
            WHERE cod_flowName = '{metrica}' 
              AND cod_runID = '{cod_runID}'
              AND (cod_processErrorCode IS NULL OR CAST(cod_processErrorCode AS STRING) = 'null')
        """
        self.spark.sql(sql_query)

    ####
    def populate_log_state(self, catalog: str, database: str, table: str, metrica: str, snapshot_date: str, cod_runID: str, is_error: bool):
        """
        input: catalog (str), database (str), table (str), metrica (str), snapshot_date (str), cod_runID (str), is_error (bool)
        description: Updates process status and end date in the log table.
        output: None
        """
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        full_table_path = f"{catalog}.{database}.{table}"
        status = "Failed" if is_error else "Succeeded"
        
        sql_query = f"""
            UPDATE {full_table_path}
            SET 
                cod_processStatus = '{status}',
                dat_processEndDate = CAST('{now}' AS TIMESTAMP)
            WHERE cod_flowName = '{metrica}' 
              AND cod_runID = '{cod_runID}'
              AND (cod_processStatus IS NULL OR CAST(cod_processStatus AS STRING) = 'null' OR cod_processStatus = 'Started')
        """
        self.spark.sql(sql_query)

    ####
    def close_process_log(self, catalog: str, database: str, table: str, metrica: str, cod_runID: str, is_error: bool, error_msg: str = None):
        """
        input: catalog (str), database (str), table (str), metrica (str), cod_runID (str), is_error (bool), error_msg (str, optional)
        description: Closes process log, updates status, end date, and error message if present.
        output: None
        """
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        full_table_path = f"{catalog}.{database}.{table}"
        
        status = "Failed" if is_error else "Succeeded"
        
        # Gestione sicura del messaggio di errore (SQL escape per gli apici singoli)
        if is_error and error_msg:
            safe_error_msg = f"'{str(error_msg).replace(chr(39), chr(39)+chr(39))}'" 
        else:
            safe_error_msg = "NULL"

        sql_query = f"""
            UPDATE {full_table_path}
            SET 
                cod_processStatus = '{status}',
                dat_processEndDate = CAST('{now}' AS TIMESTAMP),
                cod_processErrorCode = {safe_error_msg}
            WHERE cod_flowName = '{metrica}' 
            AND cod_runID = '{cod_runID}'
            AND (cod_processStatus IS NULL OR CAST(cod_processStatus AS STRING) = 'null' OR cod_processStatus = 'Started')
        """
        self.spark.sql(sql_query)
    
    ####
    def set_logging(self):
        """
        input: None
        description: Set up and return a logger with a specific format.
        output: Logger object
        """
        logger = logging.getLogger(__name__)
        logger.setLevel(logging.INFO)
        logger.propagate = False

        if not logger.handlers:

            handler = logging.StreamHandler()

            formatter = logging.Formatter(
                fmt='%(asctime)s - %(levelname)s - %(name)s:%(lineno)d - %(message)s',
                datefmt='%Y-%m-%d %H:%M:%S'
            )

            handler.setFormatter(formatter)
            logger.addHandler(handler)
        return logger

    ####
    def date_to_start_month(self, date_to_convert):
        """
        input: date_to_convert (str, format 'YYYY-MM-DD')
        description: Converts a date string to the first day of its month.
        output: Date string of the first day of the month ('YYYY-MM-DD')
        """
        dt = datetime.strptime(date_to_convert, "%Y-%m-%d")
        first_of_month = dt.replace(day=1)
        return first_of_month.strftime("%Y-%m-%d")

    ####
    def replace_month_date(self, data_str, n_month: int):
        """
        input: data_str (str, format 'YYYY-MM-DD'), n_month (int)
        description: Subtracts n_months from the given date string.
        output: Date string after subtracting n_months ('YYYY-MM-DD')
        """
        data_input = datetime.strptime(data_str, "%Y-%m-%d")

        date_mod = data_input - relativedelta(months=n_month)

        date_output = date_mod.strftime("%Y-%m-%d")
        return date_output